# LeetCode 4059 — Lexicographically Largest Power Array (Hard)

Bài toán LeetCode:

```python
class Solution:
    def largestPower(self, nums: List[int]) -> List[int]:
        ...
```

Link tra cứu: https://leetcode.com/problemset/?search=4059 (Weekly Contest 520 — câu 4, rating khoảng 2321).

Bài này kết hợp **thao tác bit** với **tham lam theo thứ tự từ điển (lexicographic greedy)**:

- "Lớn nhất theo thứ tự từ điển" nghĩa là: **ưu tiên tuyệt đối phần tử đầu tiên**, rồi mới tới phần tử thứ hai, ...
- Mỗi lần tối ưu một vị trí, ta thu được một **ràng buộc** lên hoán vị; các ràng buộc được giữ bằng cách **chia nhóm** dần dần.
- Có một "lời giải" rất hấp dẫn là **sắp mảng giảm dần** — nhưng nó **sai**! Ta sẽ tìm ra phản ví dụ và hiểu vì sao.

---

## Sau notebook này, bạn sẽ nắm được

1. Đọc bit thứ `b` của một số: `(x >> b) & 1`.
2. Định nghĩa mảng `power` và cách tính nó cho một hoán vị.
3. Brute Force thử mọi hoán vị.
4. Tham lam từ điển: tối ưu từng vị trí, giữ các nhóm "được tự do sắp" và tách nhóm khi cần.
5. Cài đặt chia nhóm tại chỗ (in-place) trên một mảng duy nhất, `O(15 · n)`.
6. Vì sao "sắp giảm dần" sai — phản ví dụ nhỏ `[5, 2, 1]`.
7. Lỗi hay gặp, mẹo nhận diện, biến thể và bài tập.

# 1. Đề bài yêu cầu gì?

Cho mảng `nums` (mỗi số `< 2^15`). Ta được sắp lại thành một hoán vị `perm` tùy ý. Từ `perm`, dựng mảng `power` dài **15**:

```text
power[i] = độ dài TIỀN TỐ dài nhất của perm mà mọi phần tử đều có bit (14 - i) bằng 1
```

(Bit đánh số từ phải sang trái, bắt đầu từ 0. Vậy `power[0]` ứng với bit cao nhất là bit 14, `power[14]` ứng với bit 0.)

Trả về mảng `power` **lớn nhất theo thứ tự từ điển** có thể đạt được.

---

## Ví dụ 1

```python
nums = [7, 5]   ->   [0,0,0,0,0,0,0,0,0,0,0,0, 2, 1, 2]
```

Chọn `perm = [7, 5]` (`7 = 111`, `5 = 101`):

- bit 2: cả hai có → `power[12] = 2`;
- bit 1: chỉ `7` có, và `7` đứng đầu → `power[13] = 1`;
- bit 0: cả hai có → `power[14] = 2`;
- các bit cao hơn: phần tử đầu không có → 0.

## Ví dụ 2

```python
nums = [3, 1, 7]   ->   [0,...,0, 1, 2, 3]
```

`perm = [7, 3, 1]`: bit 2 chỉ `7` → 1; bit 1 có `7, 3` → 2; bit 0 có cả ba → 3.

---

## Ràng buộc

```text
1 <= nums.length <= 5 * 10^4
0 <= nums[i] < 2^15
```

# 2. Kiến thức nền tảng

## 2.1. Đọc bit thứ `b`

`(x >> b) & 1` dịch `x` sang phải `b` vị trí rồi lấy bit cuối.

In [1]:
from typing import List

x = 13                       # 1101
print([(x >> b) & 1 for b in range(4)])        # bit 0..3
print(format(x, "04b"))

[1, 0, 1, 1]
1101


## 2.2. Tính `power` cho một hoán vị

Với mỗi bit từ cao xuống thấp, đếm độ dài tiền tố liên tiếp có bit đó. Ta viết hàm tổng quát với số bit `L` (đề dùng `L = 15`; ví dụ nhỏ dùng `L = 3` cho dễ nhìn).

In [2]:
def power_of(perm: List[int], L: int = 15) -> List[int]:
    res = []
    for i in range(L):
        b = L - 1 - i
        j = 0
        while j < len(perm) and (perm[j] >> b) & 1:
            j += 1
        res.append(j)
    return res


print(power_of([7, 5], 3), power_of([5, 7], 3))
print(power_of([7, 3, 1], 3), power_of([3, 7, 1], 3))

[2, 1, 2] [2, 0, 2]
[1, 2, 3] [0, 2, 3]


## 2.3. So sánh từ điển

Python so sánh `list` theo từ điển: `[1, 2, 3] > [1, 1, 9]` vì phần tử thứ hai `2 > 1` — phần tử sau **không còn ý nghĩa** một khi đã khác ở phần tử trước. Đây là tinh thần của tham lam từ điển.

In [3]:
print([1, 2, 3] > [1, 1, 9], [2, 0, 0] > [1, 9, 9])

True True


# 3. Cách 1 — Brute Force: thử mọi hoán vị

In [4]:
from itertools import permutations


def largest_power_bruteforce(nums: List[int], L: int = 15) -> List[int]:
    return max(power_of(list(p), L) for p in permutations(nums))


print(largest_power_bruteforce([7, 5], 3))
print(largest_power_bruteforce([3, 1, 7], 3))

[2, 1, 2]
[1, 2, 3]


`n!` hoán vị — chỉ dùng cho `n <= 8`.

---

# 4. Bước đột phá: tham lam từng vị trí + chia nhóm

## Vị trí đầu tiên (bit 14)

`power[0]` lớn nhất có thể = **số phần tử có bit 14**: xếp tất cả chúng lên đầu. Đây là ràng buộc bắt buộc nếu muốn đạt max (bất kỳ hoán vị nào để một phần tử không có bit 14 chen vào trước sẽ có `power[0]` nhỏ hơn).

Sau bước này, hoán vị phải có dạng: **[nhóm có bit 14] + [nhóm không có bit 14]**, và **trong mỗi nhóm được tự do sắp**.

## Vị trí tiếp theo (bit 13)

Ta tối đa `power[1]` mà **không phá** ràng buộc cũ. Duyệt các nhóm theo thứ tự:

- Nếu **mọi** phần tử của nhóm có bit 13 → cả nhóm góp vào tiền tố, sang nhóm sau.
- Ngược lại, trong nhóm này đưa các phần tử có bit 13 lên đầu nhóm → góp thêm `số phần tử có bit 13`, rồi **tiền tố dừng** (phần tử tiếp theo không có bit 13). Nhóm bị **tách** thành [có bit] + [không có bit]. Các nhóm sau không ảnh hưởng.

Lặp lại cho từng bit. Mỗi bit tách tối đa **một** nhóm.

## Mô phỏng `nums = [3, 1, 7]` với `L = 3`

| bit | các nhóm trước | nhóm dừng | power | nhóm sau |
|---:|---|---|---:|---|
| 2 | `[3, 1, 7]` | `[3,1,7]`: chỉ `7` có bit → +1, tách | 1 | `[7]`, `[3, 1]` |
| 1 | `[7]`, `[3, 1]` | `[7]` đủ → +1; `[3,1]`: chỉ `3` → +1, tách | 2 | `[7]`, `[3]`, `[1]` |
| 0 | `[7]`, `[3]`, `[1]` | cả ba đều đủ | 3 | không đổi |

Kết quả `[1, 2, 3]` ✔.

In [5]:
def largest_power_groups(nums: List[int], L: int = 15, verbose: bool = False) -> List[int]:
    groups = [nums[:]]
    res = []
    for i in range(L):
        b = L - 1 - i
        prefix = 0
        for gi, g in enumerate(groups):
            ones = [x for x in g if (x >> b) & 1]
            prefix += len(ones)
            if len(ones) == len(g):
                continue                                   # cả nhóm có bit: tiền tố đi tiếp
            if ones:                                       # nhóm hỗn hợp: tách
                groups[gi:gi + 1] = [ones, [x for x in g if not (x >> b) & 1]]
            break                                          # tiền tố dừng tại nhóm này
        res.append(prefix)
        if verbose:
            print(f"bit {b}: power = {prefix}, nhóm = {groups}")
    return res


print(largest_power_groups([3, 1, 7], 3, verbose=True))

bit 2: power = 1, nhóm = [[7], [3, 1]]
bit 1: power = 2, nhóm = [[7], [3], [1]]
bit 0: power = 3, nhóm = [[7], [3], [1]]
[1, 2, 3]


## Một quan sát quan trọng: nhóm **không bị tách** khi không ai có bit

Xét `nums = [5, 2, 1]`, `L = 3` (`5 = 101`, `2 = 010`, `1 = 001`):

| bit | các nhóm trước | chuyện gì xảy ra | power | nhóm sau |
|---:|---|---|---:|---|
| 2 | `[5, 2, 1]` | chỉ `5` có → +1, tách | 1 | `[5]`, `[2, 1]` |
| 1 | `[5]`, `[2, 1]` | `[5]` **không** có bit 1 → dừng ngay, power 0, **không tách gì** | 0 | `[5]`, `[2, 1]` |
| 0 | `[5]`, `[2, 1]` | `[5]` đủ → +1; `[2, 1]`: chỉ `1` có → +1, tách | 2 | `[5]`, `[1]`, `[2]` |

Ở bit 1, phần tử đầu (bắt buộc là `5`) không có bit 1, nên `power[1] = 0` **dù sắp `[2, 1]` thế nào**. Vì vậy ta vẫn được tự do đặt `1` trước `2` — và bit 0 tận dụng điều đó: hoán vị tối ưu là `[5, 1, 2]` với `power = [1, 0, 2]`.

In [6]:
print(largest_power_groups([5, 2, 1], 3, verbose=True))
print("power_of([5, 1, 2]) =", power_of([5, 1, 2], 3))
print("brute force         =", largest_power_bruteforce([5, 2, 1], 3))

bit 2: power = 1, nhóm = [[5], [2, 1]]
bit 1: power = 0, nhóm = [[5], [2, 1]]
bit 0: power = 2, nhóm = [[5], [1], [2]]
[1, 0, 2]
power_of([5, 1, 2]) = [1, 0, 2]
brute force         = [1, 0, 2]


# 5. Cách 3 — Chia nhóm tại chỗ trên một mảng

Cách 2 tạo list mới cho mỗi nhóm bị tách — dễ hiểu nhưng tốn bộ nhớ tạm. Cài đặt gọn hơn:

- Giữ **một** mảng `arr` (chính là hoán vị đang dựng) và danh sách `starts` = vị trí bắt đầu của từng nhóm (đoạn liên tiếp `[starts[t], starts[t+1])`).
- Ở mỗi bit, duyệt các đoạn theo thứ tự. Gặp đoạn hỗn hợp: **phân hoạch ổn định** (stable partition) đoạn đó — phần có bit lên trước — rồi chèn một điểm cắt mới vào `starts`.
- Khi xong 15 bit, `arr` là một hoán vị tối ưu thực sự (in ra được!), còn `power` tính song song.

Mỗi bit duyệt tối đa `n` phần tử → `O(15 · n)`. Mỗi bit tạo tối đa **một** nhóm mới, nên `starts` dài tối đa 17 phần tử — chèn vào nó gần như miễn phí.

In [7]:
def largest_power_inplace(nums: List[int], L: int = 15, return_perm: bool = False):
    arr = nums[:]
    n = len(arr)
    starts = [0, n]                        # đoạn t là arr[starts[t] : starts[t+1]]
    res = []
    for i in range(L):
        b = L - 1 - i
        prefix = 0
        for t in range(len(starts) - 1):
            lo, hi = starts[t], starts[t + 1]
            ones = [x for x in arr[lo:hi] if (x >> b) & 1]
            prefix += len(ones)
            if len(ones) == hi - lo:
                continue
            if ones:
                zeros = [x for x in arr[lo:hi] if not (x >> b) & 1]
                arr[lo:hi] = ones + zeros  # phân hoạch ổn định
                starts.insert(t + 1, lo + len(ones))
            break
        res.append(prefix)
    return (res, arr) if return_perm else res


res, perm = largest_power_inplace([5, 2, 1], 3, return_perm=True)
print(res, perm, power_of(perm, 3))
res, perm = largest_power_inplace([11, 4, 9, 3, 9, 2], 4, return_perm=True)
print(res, perm, power_of(perm, 4))

[1, 0, 2] [5, 1, 2] [1, 0, 2]
[3, 0, 1, 4] [11, 9, 9, 3, 4, 2] [3, 0, 1, 4]


Hoán vị in ra có `power_of(perm)` trùng đúng mảng kết quả — một cách tự kiểm tra rất thuyết phục.

# 6. Phiên bản đúng format LeetCode

Ta chỉ cần mảng `power`, nên dùng phiên bản danh sách nhóm (cách 2) cho ngắn gọn.

In [8]:
from typing import List


class Solution:
    def largestPower(self, nums: List[int]) -> List[int]:
        groups = [nums]
        res = []
        for b in range(14, -1, -1):
            prefix = 0
            for gi, g in enumerate(groups):
                ones = [x for x in g if (x >> b) & 1]
                prefix += len(ones)
                if len(ones) == len(g):
                    continue
                if ones:
                    zeros = [x for x in g if not (x >> b) & 1]
                    groups[gi:gi + 1] = [ones, zeros]
                break
            res.append(prefix)
        return res

In [9]:
sol = Solution()
tests = [
    ([7, 5], [0] * 12 + [2, 1, 2], "ví dụ 1"),
    ([3, 1, 7], [0] * 12 + [1, 2, 3], "ví dụ 2"),
    ([0], [0] * 15, "chỉ số 0"),
    ([2**15 - 1] * 3, [3] * 15, "toàn bit 1"),
    ([16384, 1], [1] + [0] * 13 + [0], "bit cao vs bit thấp"),
    ([5, 2, 1], [0] * 12 + [1, 0, 2], "nhóm không bị tách"),
    ([11, 4, 9, 3, 9, 2], [0] * 11 + [3, 0, 1, 4], "phản ví dụ sắp giảm dần"),
]
for nums, expected, name in tests:
    result = sol.largestPower(nums)
    assert result == expected, (name, result, expected)
    print(f"PASS | {name:22} -> {result}")

PASS | ví dụ 1                -> [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 2, 1, 2]
PASS | ví dụ 2                -> [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 2, 3]
PASS | chỉ số 0               -> [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
PASS | toàn bit 1             -> [3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3]
PASS | bit cao vs bit thấp    -> [1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
PASS | nhóm không bị tách     -> [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 2]
PASS | phản ví dụ sắp giảm dần -> [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 3, 0, 1, 4]


Test `[16384, 1]`: `16384 = 2^14`. Đặt nó lên đầu cho `power[0] = 1` — quan trọng hơn mọi thứ phía sau; khi đó bit 0 có tiền tố 0 (phần tử đầu không có bit 0). Nếu đặt `1` lên đầu thì `power = [0, ..., 0, 1]` — nhỏ hơn theo từ điển.

# 7. Vì sao đúng?

1. **Tham lam từ điển:** để `power` lớn nhất theo từ điển, phải tối đa `power[0]` trước; trong các hoán vị đạt max đó, tối đa `power[1]`, ...
2. **Bất biến của nhóm:** sau khi xét các bit cao, tập hoán vị đạt max cho các vị trí đã xét chính là: nối các nhóm theo thứ tự cố định, mỗi nhóm sắp tùy ý. Tách nhóm hỗn hợp (bit 1 lên trước) là điều kiện **cần và đủ** để đạt max cho bit hiện tại.
3. **Nhóm không ai có bit:** khi phần tử đầu tiên có thể đặt ở vị trí tiếp theo đều thiếu bit, `power` của bit đó bằng giá trị đã tích lũy **bất kể** sắp các nhóm còn lại thế nào — nên ta **không** thêm ràng buộc mới. Thêm ràng buộc thừa (như "sắp giảm dần" làm) có thể làm mất tối ưu ở các bit sau.
4. Mỗi bit chỉ duyệt các nhóm tới chỗ dừng, nên mỗi bước `O(n)`, tổng `O(15 · n)`.

# 8. Độ phức tạp

| Cách | Ý tưởng | Thời gian | Bộ nhớ |
|---|---|---:|---:|
| 1. Brute Force | mọi hoán vị | `O(n! · 15 · n)` | `O(n)` |
| 2. Chia nhóm | tách tối đa 1 nhóm mỗi bit | `O(15 · n)` | `O(n)` |
| 3. Chia nhóm tại chỗ | một mảng + điểm cắt | `O(15 · n)` | `O(n)` |

# 9. Đối chiếu ba cách

In [10]:
import random

random.seed(4059)
for _ in range(2000):
    nums = [random.randint(0, 15) for _ in range(random.randint(1, 6))]
    expected = largest_power_bruteforce(nums, 4)
    assert largest_power_groups(nums, 4) == expected, nums
    res, perm = largest_power_inplace(nums, 4, return_perm=True)
    assert res == expected and power_of(perm, 4) == expected, nums
for _ in range(300):
    nums = [random.randint(0, 2**15 - 1) for _ in range(random.randint(1, 200))]
    assert largest_power_groups(nums) == largest_power_inplace(nums) == sol.largestPower(nums)
print("Khớp brute force (4 bit) trên 2000 test; các cách nhanh khớp nhau trên 300 test 15 bit")

Khớp brute force (4 bit) trên 2000 test; các cách nhanh khớp nhau trên 300 test 15 bit


# 10. Những lỗi dễ mắc

## Lỗi 1 — "Sắp giảm dần là xong"

Rất hấp dẫn: số có bit cao bằng 1 thì lớn hơn, nên sắp giảm dần có vẻ "đưa bit 1 lên trước" ở mọi bit. Nhưng như mục 4 đã thấy, sắp giảm dần thêm **ràng buộc thừa** ở những bit mà `power` đã bị chặn (ví dụ bit 1 của `[5, 2, 1]`), làm mất tự do cho các bit sau.

In [11]:
def wrong_sorted_desc(nums, L=15):
    return power_of(sorted(nums, reverse=True), L)


for nums, L in [([5, 2, 1], 3), ([11, 4, 9, 3, 9, 2], 4)]:
    print(nums, "| sai:", wrong_sorted_desc(nums, L), "| đúng:", largest_power_groups(nums, L))

[5, 2, 1] | sai: [1, 0, 1] | đúng: [1, 0, 2]
[11, 4, 9, 3, 9, 2] | sai: [3, 0, 1, 3] | đúng: [3, 0, 1, 4]


## Lỗi 2 — Đi tiếp qua một nhóm **không có** phần tử nào mang bit

Nếu nhóm đầu tiên không có phần tử nào có bit `b`, tiền tố đã dừng ở **0** phần tử của nhóm đó. Code sai coi nhóm "đồng nhất" (toàn 0) giống nhóm "toàn 1" và đi tiếp, cộng nhầm các phần tử của nhóm sau.

In [12]:
def wrong_continue_on_zero(nums, L=15):
    groups = [nums[:]]
    res = []
    for i in range(L):
        b = L - 1 - i
        prefix = 0
        for gi, g in enumerate(groups):
            ones = [x for x in g if (x >> b) & 1]
            prefix += len(ones)
            if 0 < len(ones) < len(g):
                groups[gi:gi + 1] = [ones, [x for x in g if not (x >> b) & 1]]
                break
            # SAI: nhóm toàn 0 cũng "đi tiếp"
        res.append(prefix)
    return res


random.seed(1)
for _ in range(5000):
    nums = [random.randint(0, 15) for _ in range(random.randint(2, 5))]
    if wrong_continue_on_zero(nums, 4) != largest_power_bruteforce(nums, 4):
        print("Phản ví dụ:", nums, "| sai:", wrong_continue_on_zero(nums, 4), "| đúng:", largest_power_bruteforce(nums, 4))
        break

Phản ví dụ: [2, 8, 3] | sai: [1, 0, 2, 1] | đúng: [1, 0, 0, 0]


## Lỗi 3 — Tối ưu từng bit độc lập

Đếm "số phần tử có bit `b`" cho mọi bit rồi trả về luôn: mỗi bit được tối ưu với **một hoán vị khác nhau** — nhưng ta chỉ được chọn **một** hoán vị.

In [13]:
def wrong_independent(nums, L=15):
    return [sum((x >> (L - 1 - i)) & 1 for x in nums) for i in range(L)]


print("Sai :", wrong_independent([5, 2, 1], 3), "| Đúng:", largest_power_groups([5, 2, 1], 3))

Sai : [1, 1, 2] | Đúng: [1, 0, 2]


## Lỗi 4 — Đánh số bit ngược

`power[0]` ứng với **bit 14** (cao nhất), không phải bit 0. Duyệt bit từ thấp lên cao sẽ tối ưu sai thứ tự ưu tiên.

## Lỗi 5 — Brute Force

`n` tới `5·10^4` — hoán vị là không tưởng.

# 11. Mẹo nhận diện

1. "Lớn nhất / nhỏ nhất theo **thứ tự từ điển**" của một **mảng kết quả** → tham lam: tối ưu phần tử đầu, cố định ràng buộc, rồi phần tử tiếp theo.
2. Ràng buộc kiểu "các phần tử này phải đứng trước" → giữ **các nhóm có thứ tự, trong nhóm tự do**, tách nhóm dần (tinh chỉnh phân hoạch — partition refinement).
3. Cẩn thận với lối tắt "chỉ cần sắp xếp": nó đúng khi **mọi** bước đều ép thứ tự, nhưng sai khi có bước "không quan trọng" (như nhóm toàn 0 ở đây). Luôn đối chiếu brute force trước khi tin một lối tắt.

Các bài liên quan: *Largest Number* (LeetCode 179 — sắp xếp với phép so sánh tùy biến), *Maximum XOR of Two Numbers in an Array* (421 — xét bit từ cao xuống), *Lexicographically Smallest String After Applying Operations* (1625).

# 12. Biến thể

## Biến thể 1 — Trả về chính hoán vị tối ưu

Nhiều bài yêu cầu in ra hoán vị chứ không chỉ mảng `power`. Cách 3 (chia nhóm tại chỗ) làm được ngay; với danh sách nhóm của cách 2, chỉ cần **nối các nhóm** theo thứ tự lúc kết thúc. (Bài tập 6.)

## Biến thể 1b — Mảng `power` **nhỏ nhất** theo từ điển

Nghe giống nhưng khó hơn hẳn: muốn `power[i] = 0` chỉ cần **phần tử đầu** thiếu bit, nên ràng buộc không còn dạng "chia nhóm" gọn gàng — đây là ví dụ cho thấy mỗi biến thể cần kiểm chứng lại bằng brute force.

## Biến thể 2 — Số bit `L` tùy ý

Hàm `largest_power_groups(nums, L)` đã tổng quát.

In [14]:
print(largest_power_groups([5, 6, 7, 1], 3), largest_power_groups([5, 6, 7, 1], 5))

[3, 2, 1] [0, 0, 3, 2, 1]


# 13. Template tư duy

```python
# Tham lam từ điển với tinh chỉnh nhóm
groups = [all_items]
for position in order_of_importance:
    duyệt groups theo thứ tự:
        nhóm thỏa hết -> cộng, đi tiếp
        nhóm hỗn hợp  -> đưa phần thỏa lên trước (tách), cộng, dừng
        nhóm không ai thỏa -> dừng
```

# 14. Bài tập vận dụng

## Bài 1 — Tính bằng tay

`nums = [6, 5, 3]` với `L = 3`. Mô phỏng cách chia nhóm. Kiểm tra bằng cell dưới.

In [15]:
# Kiểm tra Bài 1
print(largest_power_groups([6, 5, 3], 3, verbose=True))

bit 2: power = 2, nhóm = [[6, 5], [3]]
bit 1: power = 1, nhóm = [[6], [5], [3]]
bit 0: power = 0, nhóm = [[6], [5], [3]]
[2, 1, 0]


## Bài 2 — Dự đoán output

```python
print(power_of([6, 4, 2], 3), power_of([2, 4, 6], 3))
print(largest_power_groups([6, 4, 2], 3))
```

In [16]:
print(power_of([6, 4, 2], 3), power_of([2, 4, 6], 3))
print(largest_power_groups([6, 4, 2], 3))

[2, 1, 0] [0, 1, 0]
[2, 1, 0]


## Bài 3 — Hoàn thiện `power_of`

In [17]:
def exercise_3_power_of(perm: List[int], L: int = 15) -> List[int]:
    # TODO: với mỗi bit từ L-1 xuống 0, đếm tiền tố liên tiếp có bit đó
    raise NotImplementedError("Hãy tự hoàn thiện trước khi xem đáp án.")


# assert exercise_3_power_of([7, 5], 3) == [2, 1, 2]

## Bài 4 — Largest Number (LeetCode 179)

Sắp các số để nối lại thành số lớn nhất (trả về chuỗi).

```python
largest_number([3, 30, 34, 5, 9])   -> "9534330"
```

In [18]:
def exercise_4_largest_number(nums: List[int]) -> str:
    # TODO
    raise NotImplementedError


# assert exercise_4_largest_number([3, 30, 34, 5, 9]) == "9534330"

## Bài 5 — Tiền tố có AND khác 0 dài nhất

Sắp lại `nums` để tiền tố dài nhất có **AND khác 0** là dài nhất. Trả về độ dài đó.

```python
longest_nonzero_and_prefix([3, 5, 6, 8])   -> 2    # ví dụ [3, 5] (AND = 1) hoặc [5, 6] (AND = 4)
```

Gợi ý: tiền tố có AND khác 0 ⇔ mọi phần tử chung một bit nào đó.

In [19]:
def exercise_5_longest_nonzero_and_prefix(nums: List[int]) -> int:
    # TODO
    raise NotImplementedError


# assert exercise_5_longest_nonzero_and_prefix([3, 5, 6, 8]) == 2

## Bài 6 — Mini challenge: trả về một hoán vị tối ưu

Dùng **danh sách nhóm** (cách 2, không dùng cách 3) để trả về một hoán vị `perm` của `nums` sao cho `power_of(perm, L)` là lớn nhất theo từ điển.

In [20]:
def exercise_6_best_perm(nums: List[int], L: int = 15) -> List[int]:
    # TODO
    raise NotImplementedError

## Gợi ý (mở từng mức)

**Bài 3**

- Mức 1: `b = L - 1 - i`.
- Mức 2: tăng `j` khi `(perm[j] >> b) & 1`.
- Mức 3: `res.append(j)`.

**Bài 4**

- Mức 1: so sánh hai chuỗi `a`, `b` bằng `a + b` vs `b + a`.
- Mức 2: `functools.cmp_to_key`.
- Mức 3: trường hợp toàn số 0 → `"0"`.

**Bài 5**

- Mức 1: với mỗi bit, đếm số phần tử có bit đó.
- Mức 2: đưa các phần tử đó lên đầu → tiền tố có AND chứa bit này.
- Mức 3: đáp án = `max` số đếm trên mọi bit.

**Bài 6**

- Mức 1: chạy đúng vòng lặp của `largest_power_groups`.
- Mức 2: thứ tự các nhóm trong `groups` luôn là thứ tự trong hoán vị.
- Mức 3: cuối cùng trả về `[x for g in groups for x in g]`.

# 15. Đáp án tham khảo

## Đáp án Bài 1

`6 = 110`, `5 = 101`, `3 = 011`.

| bit | nhóm | power | sau |
|---:|---|---:|---|
| 2 | `[6, 5, 3]`: `6, 5` có → +2, tách | 2 | `[6, 5]`, `[3]` |
| 1 | `[6, 5]`: chỉ `6` có → +1, tách | 1 | `[6]`, `[5]`, `[3]` |
| 0 | `[6]` không có bit 0 → dừng | 0 | |

Kết quả `[2, 1, 0]`.

## Đáp án Bài 2

```text
[2, 1, 0] [0, 1, 0]
[2, 1, 0]
```

`[2, 4, 6]`: phần tử đầu `2 = 010` không có bit 2 → 0; bit 1: `2` có, `4 = 100` không → 1; bit 0: `2` không có → 0.

In [21]:
# Đáp án Bài 3
def answer_3_power_of(perm: List[int], L: int = 15) -> List[int]:
    return power_of(perm, L)


assert answer_3_power_of([7, 5], 3) == [2, 1, 2]
print("Bài 3: PASS")

Bài 3: PASS


In [22]:
# Đáp án Bài 4
from functools import cmp_to_key


def answer_4_largest_number(nums: List[int]) -> str:
    strs = sorted(map(str, nums), key=cmp_to_key(lambda a, b: -1 if a + b > b + a else (1 if a + b < b + a else 0)))
    res = "".join(strs)
    return "0" if res[0] == "0" else res


assert answer_4_largest_number([3, 30, 34, 5, 9]) == "9534330"
assert answer_4_largest_number([0, 0]) == "0"
print("Bài 4: PASS")

Bài 4: PASS


In [23]:
# Đáp án Bài 5
def answer_5_longest_nonzero_and_prefix(nums: List[int]) -> int:
    return max(sum((x >> b) & 1 for x in nums) for b in range(15))


def brute_and_prefix(nums):
    best = 0
    for p in permutations(nums):
        acc = None
        for j, x in enumerate(p):
            acc = x if acc is None else acc & x
            if acc == 0:
                break
            best = max(best, j + 1)
    return best


assert answer_5_longest_nonzero_and_prefix([3, 5, 6, 8]) == 2
random.seed(5)
for _ in range(500):
    arr = [random.randint(0, 15) for _ in range(random.randint(1, 6))]
    assert answer_5_longest_nonzero_and_prefix(arr) == brute_and_prefix(arr)
print("Bài 5: PASS")

Bài 5: PASS


In [24]:
# Đáp án Bài 6
def answer_6_best_perm(nums: List[int], L: int = 15) -> List[int]:
    groups = [nums[:]]
    for i in range(L):
        b = L - 1 - i
        for gi, g in enumerate(groups):
            ones = [x for x in g if (x >> b) & 1]
            if len(ones) == len(g):
                continue
            if ones:
                groups[gi:gi + 1] = [ones, [x for x in g if not (x >> b) & 1]]
            break
    return [x for g in groups for x in g]


print(answer_6_best_perm([5, 2, 1], 3), answer_6_best_perm([11, 4, 9, 3, 9, 2], 4))
random.seed(6)
for _ in range(2000):
    arr = [random.randint(0, 15) for _ in range(random.randint(1, 6))]
    perm = answer_6_best_perm(arr, 4)
    assert sorted(perm) == sorted(arr)
    assert power_of(perm, 4) == largest_power_bruteforce(arr, 4), arr
print("Bài 6: PASS (hoán vị hợp lệ, khớp brute force)")

[5, 1, 2] [11, 9, 9, 3, 4, 2]


Bài 6: PASS (hoán vị hợp lệ, khớp brute force)


# 16. Tổng kết

**Lexicographically Largest Power Array**:

1. Tham lam từ điển: tối đa `power[0]` (bit 14), rồi `power[1]`, ... mà không phá ràng buộc cũ.
2. Ràng buộc giữ bằng các nhóm có thứ tự; mỗi bit tách tối đa một nhóm; nhóm không có ai mang bit thì tiền tố dừng.
3. **Không** dùng lối tắt sắp giảm dần — nó thêm ràng buộc thừa (phản ví dụ `[5, 2, 1]`).
4. Cài đặt tại chỗ cho luôn một hoán vị tối ưu để tự kiểm tra.

## Độ phức tạp

```text
Time:  O(15 · n)
Space: O(n)
```

# 17. Checklist trước khi submit

- [ ] `power[i]` ứng với bit `14 - i` (cao xuống thấp)?
- [ ] Hoán vị là **một** cho mọi bit (không tối ưu từng bit riêng)?
- [ ] Nhóm không có phần tử mang bit → tiền tố **dừng**?
- [ ] Không dùng "sắp giảm dần" (phản ví dụ `[5, 2, 1]`)?
- [ ] Đã test: số 0, toàn bit 1, bit cao chỉ một phần tử?